# Балансировка коленчатого вала — демонстрация

Ноутбук использует пакет `crankshaft-balancing` (`pip install -e .` из корня репозитория).
Вся логика находится в `src/crankshaft_balancing/`, здесь — только вызовы и графики.
Теория — в `docs/theory.md`. Разделы 1–9 повторяют исходный ноутбук, раздел 10 —
новые возможности (ограничения на противовесы, свои позиции противовесов).

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from crankshaft_balancing import (
    CounterweightConstraints, analyze, build_geometry, load_shaft,
    make_initial_coefficients, optimize_counterweights,
    optimize_counterweights_history, section_grid, sweep_sensitivity,
    verify_kkt_conditions, verify_optimum_via_duality, NMM_TO_NM)
from crankshaft_balancing.analysis import raw_moments
from crankshaft_balancing.plotting import (plot_convergence, plot_distribution,
                                           plot_moment_3d, plot_sensitivity)

## 1. Исходные данные и расчёт
Замените `example_shaft.json` на свой файл — формат описан в README.

In [ ]:
engine, components = load_shaft("../examples/example_shaft.json")
res = analyze(engine, components, step=0.01)
f, Z, L = res.forces, res.Z, res.engine.crankshaft_length

Порядок элементов во входном списке не важен — при любой перестановке геометрия одна и та же:

In [ ]:
ref = build_geometry(components)
shuffled = components.copy()
random.Random(0).shuffle(shuffled)
g = build_geometry(shuffled)
print("центры щёк совпадают:   ", np.allclose(ref.cheek_centers, g.cheek_centers))
print("центры шеек совпадают:  ", np.allclose(ref.crankpin_centers, g.crankpin_centers))

## 2. Эпюры момента до оптимизации

In [ ]:
Mx_raw, My_raw = raw_moments(Z, f)
groups = lambda Fc, Fp: [{"L": f.L_cheeks, "F": Fc, "name": "Cheek"},
                         {"L": f.L_crankpins, "F": Fp, "name": "crankpin"}]
plot_distribution(Z, Mx_raw, L, groups=groups(f.Fx_cheeks, f.Fx_crankpins),
                  title="Момент M(z) до оптимизации (ось X)", ylabel="Момент, Н·м")
plot_distribution(Z, My_raw, L, groups=groups(f.Fy_cheeks, f.Fy_crankpins),
                  title="Момент M(z) до оптимизации (ось Y)", ylabel="Момент, Н·м")

## 3. После подбора противовесов

In [ ]:
for axis, r, Fc, Fp in (("Y", res.result_y, f.Fy_cheeks, f.Fy_crankpins),
                        ("X", res.result_x, f.Fx_cheeks, f.Fx_crankpins)):
    plot_distribution(
        Z, r["M"] * NMM_TO_NM, L,
        groups=groups(Fc, Fp) + [{"L": res.L_counterweights, "F": r["F3"], "name": "F3 (оптим.)"}],
        title=f"Момент M(z) после оптимизации F3 (ось {axis})", ylabel="Момент, Н·м")

## 3.1. Таблица статических моментов противовесов (новое)
Компоненты `s_x` (ось y JSON), `s_y` (ось z JSON), суммарный модуль и угол от оси z.

In [ ]:
from crankshaft_balancing import counterweight_rows, format_counterweight_table
from crankshaft_balancing.plotting import plot_counterweight_table

rows = counterweight_rows(res)
print(format_counterweight_table(rows))   # lang="en" — English labels
plot_counterweight_table(rows)            # the same table as a picture


## 4. Суммарный момент (статичный и интерактивный 3D-график)

In [ ]:
plot_moment_3d(Z, res.result_x, res.result_y, shaft_length=L, interactive=False)

In [ ]:
# Интерактивная версия (нужен plotly: pip install -e ".[interactive]"):
# plot_moment_3d(Z, res.result_x, res.result_y, shaft_length=L,
#                interactive=True, save_path="moment_3d.html")

## 5. Сходимость симплекс-метода

Использует устаревший `linprog(method="simplex")` — доступен не во всех версиях SciPy
и работает на грубой сетке (шаг 2 мм). По оси X момент нулевой по построению геометрии,
поэтому без границ на `s` задача вырожденная для симплекса («unbounded») — как и в
исходном ноутбуке, ставим границы.

In [ ]:
Z_coarse = section_grid(L, step=2.0)

history_y = optimize_counterweights_history(
    Z_coarse, f.L_cheeks, f.Fy_cheeks, f.L_crankpins, f.Fy_crankpins,
    engine.angular_speed, make_initial_coefficients(len(f.Fy_cheeks)), axis="y")
plot_convergence(history_y["history"], axis="y")

In [ ]:
history_x = optimize_counterweights_history(
    Z_coarse, f.L_cheeks, f.Fx_cheeks, f.L_crankpins, f.Fx_crankpins,
    engine.angular_speed, make_initial_coefficients(len(f.Fx_cheeks)),
    s_bounds=[(-5, 5)] * len(f.Fx_cheeks), axis="x")
plot_convergence(history_x["history"], axis="x")

## 6. Проверка, что минимум глобальный — ось Y

In [ ]:
verify_optimum_via_duality(res.result_y, axis="y")
print()
verify_kkt_conditions(res.result_y, axis="y")

In [ ]:
curves_y = sweep_sensitivity(res.result_y, Z, f.L_cheeks, f.Fy_cheeks,
                             f.L_crankpins, f.Fy_crankpins, engine.angular_speed)
plot_sensitivity(curves_y, res.result_y["s"], axis="y")

## 7. Проверка — ось X (тривиальный случай: момент и так почти нулевой)

In [ ]:
verify_optimum_via_duality(res.result_x, axis="x")
print()
verify_kkt_conditions(res.result_x, axis="x")

In [ ]:
curves_x = sweep_sensitivity(res.result_x, Z, f.L_cheeks, f.Fx_cheeks,
                             f.L_crankpins, f.Fx_crankpins, engine.angular_speed)
plot_sensitivity(curves_x, res.result_x["s"], axis="x")

## 8. Низкоуровневый вызов оптимизации

Те же параметры, что в исходном ноутбуке: `s_init`, `s_fixed`, `s_bounds`, `L3`.
Значение `"F"` в `s_init` фиксирует коэффициент на нуле (противовеса на щеке нет).

In [ ]:
s_init = make_initial_coefficients(len(f.Fy_cheeks), overrides={2: "F", 5: "F"})
r = optimize_counterweights(Z, f.L_cheeks, f.Fy_cheeks, f.L_crankpins, f.Fy_crankpins,
                            engine.angular_speed, s_init, axis="y")

## 9. Ограничения на противовесы через `analyze()` (новое)

Номер противовеса — номер щеки вдоль вала по возрастанию `x`, с 0 (список печатается
при расчёте). Ключ `"all"` действует на обе оси, `"x"`/`"y"` — на одну.

In [ ]:
constraints = {
    "all": CounterweightConstraints.from_dict({"zero": [2, 5]}),         # убрать на щеках 2 и 5
    "y":   CounterweightConstraints.from_dict({
        "fixed":  {"3": 0.2},                                            # задано вручную
        "bounds": {"0": [-0.15, 0.15], "7": [-0.15, 0.15]}}),            # ограничить диапазон
}
res_c = analyze(engine, components, step=0.05, constraints=constraints)
print("s (Y):", np.round(res_c.result_y["s"], 4))
print(f"max|My|: {res.max_moment_y:.2f} Н·м без ограничений -> {res_c.max_moment_y:.2f} Н·м с ограничениями")

## 10. Свои позиции противовесов (новое)
Противовесы можно поставить не только на щёки: координаты `x` в мм вдоль вала.

In [ ]:
positions = [31.469, 99.531, 262.0, 424.469, 492.531]
res_p = analyze(engine, components, step=0.05, counterweight_positions=positions)
print("s (Y):", np.round(res_p.result_y["s"], 4))